# PhoBERT v2: train on cleaned data

Same setup as `train_phobert_kaggle.ipynb` (GPU T4 x2, Internet on, secret `HF_TOKEN`), but the
input dataset is **`vfyr-clean-splits.zip`** which contains two folders:

- `dedup/` – duplicates and train→val/test leakage removed
- `clean/` – `dedup` + likely label errors (confident learning) removed from **train only**

Val/test are identical in both, so the two runs are directly comparable.
Both runs use **no class weights** (v1 over-predicted *offensive*) and up to 6 epochs with early stopping.

In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji
import glob, os, shutil, json
for variant in ["dedup", "clean"]:
    os.makedirs(f"data/{variant}", exist_ok=True)
    for split in ["train", "val", "test"]:
        src = glob.glob(f"/kaggle/input/**/{variant}/{split}.csv", recursive=True)
        assert len(src) == 1, (variant, split, src)
        shutil.copy(src[0], f"data/{variant}/{split}.csv")
print(os.listdir("data/dedup"), os.listdir("data/clean"))

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
# Run C: dedup data, no class weights
!python scripts/train_phobert.py --data-dir data/dedup --no-class-weights --epochs 6 --output-dir models/C --tag _dedup_nocw --no-push

In [ ]:
# Run D: dedup + pruned label errors, no class weights
!python scripts/train_phobert.py --data-dir data/clean --no-class-weights --epochs 6 --output-dir models/D --tag _clean_nocw --no-push

In [ ]:
c = json.load(open("reports/phobert_results_dedup_nocw.json"))
d = json.load(open("reports/phobert_results_clean_nocw.json"))
for name, r in [("C dedup", c), ("D dedup+pruned", d)]:
    print(f"{name}: val F1={r['val_f1_macro']} | test F1={r['test']['f1_macro']} per-class={r['test']['f1_per_class']}")
best = "models/D/best" if d["val_f1_macro"] >= c["val_f1_macro"] else "models/C/best"
print("Pushing", best)
from huggingface_hub import HfApi
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
HfApi().upload_folder(folder_path=best, repo_id=repo_id, commit_message=f"v2: trained on {best}")
print(f"https://huggingface.co/{repo_id}")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps({"C_dedup_nocw": c, "D_clean_nocw": d}, ensure_ascii=False))